# Further Performance Metrics (SMOTE dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
X_train = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/X_train_final_smote.csv")
X_test = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/X_test_final.csv")
X_val = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/X_val_final.csv")

y_train = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/y_train_final_smote.csv")
y_test = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/y_test_final.csv")
y_val = pd.read_csv("/content/drive/My Drive/Colab Notebooks/data/y_val_final.csv")

## 5 Combinations of Features

In [3]:
feature_sets = {
    "set_1": [
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3",
        "trust3",
        "ethical80",
        "civic53",
        "compassion1",
        "compassion2",
        "compassion3",
        "compassion4",
        "growth16",
        "age",
        "marital_status",
        "religious1",
        "income_scale"
    ],

    "set_2": [
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3",
        "trust3",
        "ethical80",
        "civic53",
        "compassion2",
        "compassion3",
        "growth16",
        "age",
        "marital_status"
    ],

    "set_3": [ # maslow needs only
        "sact",
        "physio1",
        "safety5",
        "belonging1",
        "esteem3"
    ],

    "set_4": [ # maturity of heart only
        "trust3",
        "ethical80",
        "civic53",
        "compassion1",
        "compassion2",
        "growth16"
    ],

    "set_5": [ # hybrid
        "sact",
        "physio1",
        "safety5",
        "trust3",
        "ethical80"
    ],

    "set_6": [ # hybrid top 4
        "sact",
        "physio1",
        "safety5",
        "trust3"
    ],

    "set_7": [ # top 3
        "sact",
        "physio1",
        "safety5"
    ]
}



Create data splits.

In [4]:
data_splits = {}

for name, features in feature_sets.items():
    data_splits[name] = {
        "X_train": X_train[features],
        "X_val": X_val[features],
        "X_test": X_test[features],
    }

In [5]:
import warnings
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
pd.pandas.set_option("display.max_columns", None)

In [6]:
models = [
    {
        "name": "RandomForest",
        "estimator": RandomForestClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "max_depth": [None, 8, 12],
            "min_samples_split": [2, 4, 6],
            "max_features": ['sqrt', 'log2']
        }
    },
    {
        "name": "LightGBM",
        "estimator": LGBMClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [50, 100, 150],
            "learning_rate": [0.01, 0.05, 0.1],
            "max_depth": [-1, 5, 8]
        }
    },
    {
        "name": "LogisticRegression",
        "estimator": LogisticRegression(random_state=42, max_iter=3000),
        "param_grid": {
            "penalty": ['l1', 'l2'],
            "C": [0.01, 0.1, 1, 10],
            "solver": ['liblinear', 'saga']
        }
    },
    {
        "name": "NaiveBayes",
        "estimator": GaussianNB(),
        "param_grid": {
            "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]
        }
    }
]


## Model Training

In [7]:
import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/')
import my_tools

all_results = {}
all_models = {}

for combo_name, data in data_splits.items():
    print(f"\n==============================")
    print(f"Running feature set: {combo_name}")
    print(f"==============================")

    results_df, best_models =  my_tools.train_ensemble_models(
        data["X_train"], y_train,
        data["X_val"], y_val,
        data["X_test"], y_test,
        models
    )

    all_results[combo_name] = results_df
    all_models[combo_name] = best_models


Running feature set: set_1
y_train: Converted to 1D pd automatically...
y_test: Converted to 1D pd automatically...
y_val: Converted to 1D pd automatically...

Training RandomForest...

Training LightGBM...
[LightGBM] [Info] Number of positive: 56869, number of negative: 56869
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010360 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4335
[LightGBM] [Info] Number of data points in the train set: 113738, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [8]:
with open("/content/drive/My Drive/Colab Notebooks/data/model_feature_comparison_smote.txt", "w") as f:
    for combo_name, df in all_results.items():
        f.write("\n" + "="*60 + "\n")
        f.write(f"FEATURE SET: {combo_name}, Predictors: {len(feature_sets[combo_name])}\n")
        f.write("="*60 + "\n\n")

        # ===== Metrics table =====
        f.write(df.to_string(index=False))
        f.write("\n\n")

        # ===== Classification Reports =====
        f.write("Classification Reports:\n")

        X_test_subset = data_splits[combo_name]["X_test"]

        for model_name, model_info in all_models[combo_name].items():
            model = model_info["model"] # extract trained model

            y_pred = model.predict(X_test_subset)

            f.write(f"\n\n--- {model_name} ---\n")
            f.write(classification_report(y_test, y_pred, digits=4))

        f.write("\n\n")

    print("INFO: Output generated successfully!")

INFO: Output generated successfully!


In [9]:
import joblib

best_models_per_combo = {}

for combo_name in all_results.keys():
    # Get best model name (top row)
    best_model_name = all_results[combo_name].iloc[0]["Model"]

    # Get trained model
    best_model = all_models[combo_name][best_model_name]["model"]

    # Store in dict (optional)
    best_models_per_combo[combo_name] = best_model

    # Save to file
    filename = f"/content/drive/My Drive/Colab Notebooks/data/best_model_smote_{combo_name}.pkl"
    joblib.dump(best_model, filename)

    print(f"{combo_name}: Saved {best_model_name} → {filename}")

set_1: Saved LightGBM → /content/drive/My Drive/Colab Notebooks/data/best_model_smote_set_1.pkl
set_2: Saved LightGBM → /content/drive/My Drive/Colab Notebooks/data/best_model_smote_set_2.pkl
set_3: Saved RandomForest → /content/drive/My Drive/Colab Notebooks/data/best_model_smote_set_3.pkl
set_4: Saved LightGBM → /content/drive/My Drive/Colab Notebooks/data/best_model_smote_set_4.pkl
set_5: Saved LightGBM → /content/drive/My Drive/Colab Notebooks/data/best_model_smote_set_5.pkl
set_6: Saved RandomForest → /content/drive/My Drive/Colab Notebooks/data/best_model_smote_set_6.pkl
set_7: Saved LightGBM → /content/drive/My Drive/Colab Notebooks/data/best_model_smote_set_7.pkl
